# Test MCP Servers via MaaS Gateway

This notebook tests MCP server access through the MaaS gateway:
1. MCP endpoint discovery via gateway
2. Authentication enforcement
3. SSE connectivity per server
4. Tool invocation test
5. Context7 (external) integration

**Prerequisites:**
- MaaS enabled with MCP servers registered (`2_enable_maas.ipynb` completed)
- At least one MCP server deployed in `mcp-servers` namespace

In [ ]:
import subprocess
import json
import os

result = subprocess.run(
    ["kubectl", "get", "ingresses.config.openshift.io", "cluster",
     "-o", "jsonpath={.spec.domain}"],
    capture_output=True, text=True
)
CLUSTER_DOMAIN = result.stdout.strip()
MAAS_HOST = f"https://maas.{CLUSTER_DOMAIN}"

# Use API key if available, otherwise fall back to OCP token
API_KEY = os.getenv("MAAS_API_KEY", "")
if not API_KEY:
    token_result = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
    API_KEY = token_result.stdout.strip()
    print("Using OpenShift token for authentication")
else:
    print(f"Using MaaS API key: {API_KEY[:15]}...")

print(f"\n\u2705 MaaS Gateway: {MAAS_HOST}")

## 1. Discover MCP Endpoints

List all MCP servers registered with the MaaS gateway via HTTPRoute.

In [ ]:
%%bash
MCP_NS="mcp-servers"
CLUSTER_DOMAIN=$(kubectl get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas.${CLUSTER_DOMAIN}"

echo "MCP Servers via MaaS Gateway"
echo "============================================================"
echo ""

printf "%-25s %-50s %s\n" "SERVER" "GATEWAY ENDPOINT" "STATUS"
printf "%-25s %-50s %s\n" "-------" "----------------" "------"

# Context7 (external, direct)
printf "%-25s %-50s" "context7 (external)" "https://mcp.context7.com/mcp"
HTTP_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 5 "https://mcp.context7.com/mcp")
if [ "$HTTP_CODE" = "200" ] || [ "$HTTP_CODE" = "405" ]; then
    printf " \u2705\n"
else
    printf " \u26a0\ufe0f (HTTP %s)\n" "$HTTP_CODE"
fi

# MCP servers via gateway
for route in $(kubectl get httproute -n ${MCP_NS} -l maas.opendatahub.io/managed=true -o jsonpath='{range .items[*]}{.metadata.name}{"\n"}{end}' 2>/dev/null); do
    SHORT_NAME=$(echo $route | sed 's/^mcp-route-//')
    URL="${HOST}/mcp/${SHORT_NAME}/sse"
    printf "%-25s %-50s" "${SHORT_NAME}" "${URL}"
    HTTP_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 5 \
      -H "Authorization: Bearer $(oc whoami -t)" "${URL}")
    if [ "$HTTP_CODE" = "200" ] || [ "$HTTP_CODE" = "405" ]; then
        printf " \u2705\n"
    else
        printf " \u26a0\ufe0f (HTTP %s)\n" "$HTTP_CODE"
    fi
done

## 2. Test Authentication Enforcement

Verify that MCP endpoints via the gateway require a valid API key.

In [ ]:
%%bash
MCP_NS="mcp-servers"
CLUSTER_DOMAIN=$(kubectl get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas.${CLUSTER_DOMAIN}"

# Pick first MCP route
FIRST_ROUTE=$(kubectl get httproute -n ${MCP_NS} -l maas.opendatahub.io/managed=true -o jsonpath='{.items[0].metadata.name}' 2>/dev/null)
SHORT_NAME=$(echo $FIRST_ROUTE | sed 's/^mcp-route-//')
URL="${HOST}/mcp/${SHORT_NAME}/sse"

echo "Testing auth enforcement on: ${URL}"
echo ""

# Test without auth
echo "1. No auth header (expecting 401/403):"
HTTP_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 5 "${URL}")
if [ "$HTTP_CODE" = "401" ] || [ "$HTTP_CODE" = "403" ]; then
    echo "   \u2705 Rejected (HTTP ${HTTP_CODE}) — auth enforced"
else
    echo "   \u26a0\ufe0f  Got HTTP ${HTTP_CODE} — auth may not be enforced"
fi

echo ""

# Test with invalid key
echo "2. Invalid API key (expecting 401/403):"
HTTP_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 5 \
  -H "Authorization: Bearer sk-oai-INVALID-KEY" "${URL}")
if [ "$HTTP_CODE" = "401" ] || [ "$HTTP_CODE" = "403" ]; then
    echo "   \u2705 Rejected (HTTP ${HTTP_CODE}) — invalid key rejected"
else
    echo "   \u26a0\ufe0f  Got HTTP ${HTTP_CODE}"
fi

echo ""

# Test with valid token
echo "3. Valid OCP token (expecting 200/405):"
HTTP_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 5 \
  -H "Authorization: Bearer $(oc whoami -t)" "${URL}")
if [ "$HTTP_CODE" = "200" ] || [ "$HTTP_CODE" = "405" ]; then
    echo "   \u2705 Accepted (HTTP ${HTTP_CODE}) — valid auth works"
else
    echo "   \u26a0\ufe0f  Got HTTP ${HTTP_CODE}"
fi

## 3. Test MCP Protocol — Sequential Thinking

Send an MCP `initialize` request to the Sequential Thinking server via the gateway.

In [ ]:
%%bash
CLUSTER_DOMAIN=$(kubectl get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas.${CLUSTER_DOMAIN}"
URL="${HOST}/mcp/sequential-thinking/sse"

echo "Testing MCP protocol on Sequential Thinking server..."
echo "Endpoint: ${URL}"
echo ""

# SSE connection test (timeout after 3 seconds to just verify connectivity)
echo "SSE connection test (3s timeout):"
RESPONSE=$(curl -sSk -m 3 \
  -H "Authorization: Bearer $(oc whoami -t)" \
  -H "Accept: text/event-stream" \
  "${URL}" 2>&1 || true)

if echo "$RESPONSE" | grep -q "event:\|data:"; then
    echo "\u2705 SSE stream active"
    echo "   First event:"
    echo "$RESPONSE" | head -5
else
    echo "Response: ${RESPONSE:0:200}"
fi

## 4. Test MCP Protocol — GitHub

Verify the GitHub MCP server is accessible through the gateway.

In [ ]:
%%bash
CLUSTER_DOMAIN=$(kubectl get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas.${CLUSTER_DOMAIN}"
URL="${HOST}/mcp/github/sse"

echo "Testing MCP protocol on GitHub server..."
echo "Endpoint: ${URL}"
echo ""

# SSE connection test
echo "SSE connection test (3s timeout):"
RESPONSE=$(curl -sSk -m 3 \
  -H "Authorization: Bearer $(oc whoami -t)" \
  -H "Accept: text/event-stream" \
  "${URL}" 2>&1 || true)

if echo "$RESPONSE" | grep -q "event:\|data:"; then
    echo "\u2705 SSE stream active"
    echo "   First event:"
    echo "$RESPONSE" | head -5
else
    echo "Response: ${RESPONSE:0:200}"
fi

## 5. Test MCP Protocol — gh-grep

Verify the gh-grep MCP server is accessible through the gateway.

In [ ]:
%%bash
CLUSTER_DOMAIN=$(kubectl get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas.${CLUSTER_DOMAIN}"
URL="${HOST}/mcp/gh-grep/sse"

echo "Testing MCP protocol on gh-grep server..."
echo "Endpoint: ${URL}"
echo ""

# SSE connection test
echo "SSE connection test (3s timeout):"
RESPONSE=$(curl -sSk -m 3 \
  -H "Authorization: Bearer $(oc whoami -t)" \
  -H "Accept: text/event-stream" \
  "${URL}" 2>&1 || true)

if echo "$RESPONSE" | grep -q "event:\|data:"; then
    echo "\u2705 SSE stream active"
    echo "   First event:"
    echo "$RESPONSE" | head -5
else
    echo "Response: ${RESPONSE:0:200}"
fi

## 6. Test MCP Protocol — Chrome DevTools

Verify the Chrome DevTools MCP server is accessible through the gateway.

In [ ]:
%%bash
CLUSTER_DOMAIN=$(kubectl get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas.${CLUSTER_DOMAIN}"
URL="${HOST}/mcp/chrome-devtools/sse"

echo "Testing MCP protocol on Chrome DevTools server..."
echo "Endpoint: ${URL}"
echo ""

# SSE connection test
echo "SSE connection test (3s timeout):"
RESPONSE=$(curl -sSk -m 3 \
  -H "Authorization: Bearer $(oc whoami -t)" \
  -H "Accept: text/event-stream" \
  "${URL}" 2>&1 || true)

if echo "$RESPONSE" | grep -q "event:\|data:"; then
    echo "\u2705 SSE stream active"
    echo "   First event:"
    echo "$RESPONSE" | head -5
else
    echo "Response: ${RESPONSE:0:200}"
fi

## 7. Comparison: Direct Route vs MaaS Gateway

Compare accessing MCP servers directly via OpenShift Routes vs through the MaaS gateway.

In [ ]:
%%bash
MCP_NS="mcp-servers"
CLUSTER_DOMAIN=$(kubectl get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
MAAS_HOST="https://maas.${CLUSTER_DOMAIN}"

echo "Direct Route vs MaaS Gateway"
echo "============================================================"
echo ""
printf "%-25s %-12s %-12s\n" "SERVER" "DIRECT" "VIA MAAS"
printf "%-25s %-12s %-12s\n" "-------" "------" "--------"

for route in $(oc get routes -n ${MCP_NS} -o jsonpath='{range .items[*]}{.metadata.name}{"\n"}{end}' 2>/dev/null); do
    host=$(oc get route $route -n ${MCP_NS} -o jsonpath='{.spec.host}')
    DIRECT_URL="https://${host}/sse"
    SHORT_NAME=$(echo $route | sed 's/^mcp-//')
    MAAS_URL="${MAAS_HOST}/mcp/${SHORT_NAME}/sse"

    # Direct access (no auth)
    DIRECT_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 5 "${DIRECT_URL}")

    # MaaS access (no auth — should be blocked)
    MAAS_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 5 "${MAAS_URL}")

    printf "%-25s HTTP %-6s  HTTP %-6s\n" "${route}" "${DIRECT_CODE}" "${MAAS_CODE}"
done

echo ""
echo "Expected: Direct routes return 200/405 (open), MaaS routes return 401/403 (auth required)"
echo ""
echo "\u2705 MaaS gateway enforces authentication on MCP tool access"

## Summary

| Test | What It Validates |
|------|-------------------|
| Endpoint Discovery | MCP servers registered via HTTPRoute and accessible |
| Auth Enforcement | Gateway rejects unauthenticated/invalid MCP requests |
| SSE Connectivity | MCP protocol (Server-Sent Events) works through gateway |
| Direct vs Gateway | MaaS adds auth layer vs open direct Routes |

### MCP Server Endpoints (via MaaS Gateway)

| Server | Gateway Path |
|--------|--------------|
| Context7 | `https://mcp.context7.com/mcp` (external, direct) |
| Sequential Thinking | `https://maas.<domain>/mcp/sequential-thinking/sse` |
| GitHub | `https://maas.<domain>/mcp/github/sse` |
| gh-grep | `https://maas.<domain>/mcp/gh-grep/sse` |
| Chrome DevTools | `https://maas.<domain>/mcp/chrome-devtools/sse` |

## Next Steps

\u2192 `5_ide_configuration.ipynb` \u2014 Configure your IDE to use MaaS for both models and MCP tools
\u2192 `6_maas_advanced.ipynb` \u2014 Explore subscriptions, rate limit tuning, and monitoring